# Normalizing Data and Generating Insights from a Project Monitoring Database

## Project Overview

This notebook transforms a single project monitoring master table containing 80+ columns into a set of normalized relational tables following database normalization principles.

The original dataset serves as the master source of project information. Since some operational tables (e.g., contribution records, time logs, legal activities, and plan history) are not available in the original dataset, realistic mock data is generated to simulate how these tables would function in an actual project management database.

The objectives of this project are to:

- Normalize the master table into multiple relational tables.
- Generate synthetic transactional data based on realistic business rules.
- Maintain referential integrity between tables.
- Perform exploratory analysis and generate business insights from the normalized database.

> **Note:** Due to data confidentiality, mock data is used for several transactional tables while preserving the structure and relationships of the original project data.

## Read Data

In [42]:
import pandas as pd
import altair as alt
import numpy as np
import re
import random
from datetime import timedelta

random.seed(42)
np.random.seed(42)

In [43]:
term_df = pd.read_csv("data/intern(term).csv", skiprows = 5)
term_df.head()

,Job \nCode,Client \nCode,Year,PD,PM,WS,Client,Subject,T,Fee,...,NAW-D.1,HZP-D.1,ABH-D.1,DTP-D.1,IFI-D.1,IMC-D.1,HZR-D.1,Service Group,Count Check,Jobcode V2
0,BCDR007,KPK01,2013,AOW,NVN,2,Komisi Pemberantasan Korupsi,BCP Development (training dan menyusun BIA),t1,"123,690,000",...,-,-,-,-,-,-,-,NaN,NaN,NaN
1,BCDR008,IDS01,2013,AOW,MSM,2,Indosat Ooredoo,Consultant services for BCM Enhancement,t1,"199,200,000",...,-,-,-,-,-,-,-,NaN,NaN,NaN
2,BCDR009,BCI01,2014,TSR,SHS,2,Bumiputera Sekuritas,BCP Review,t1,"8,000,000",...,-,-,-,-,-,-,-,NaN,NaN,NaN
3,BCDR009,BCI01,2014,TSR,SHS,2,Bumiputera Sekuritas,BCP Review,t2,"12,000,000",...,-,-,-,-,-,-,-,NaN,NaN,NaN
4,BCDR010,BNI01,2014,TSR,NVN,2,BNI Securities,BCP Review & VA,t1,"19,000,000",...,-,-,-,-,-,-,-,NaN,NaN,NaN


## Data Cleaning

In [44]:
df = term_df.copy()

In [45]:
#Rename Columns

df = df.rename(columns={
    'Job \nCode': 'Job Code',
    'Client \nCode': 'Client Code',
    'T': 'Term',
    'WS': 'Work Status',
    'BAST #': 'BAST Status',
    'Kontrak #': 'Contract Available',
    'BAST Sent (yyyy-mm-dd)': 'BAST Sent',
    'BAST Received (yyyy-mm-dd)': 'BAST Received',
    'Invoice Sent (yyyy-mm-dd)': 'Invoice Sent',
    'Payment Estimate (yyyy-mm-dd)': 'Payment Estimate',
    'Payment Received (yyyy-mm-dd)': 'Payment Received',
    'Baseline delivery (yyyy-mm-dd)': 'Baseline delivery Date',
    'Plan delivery (yyyy-mm-dd)': 'Plan delivery Date',
    'Delayed (Days)': 'Delayed Days',
    'Last Status': 'Latest Update'

})

df.columns.tolist()

['Job Code',
 'Client Code',
 'Year',
 'PD',
 'PM',
 'Work Status',
 'Client',
 'Subject',
 'Term',
 'Fee',
 'SPK Date',
 'Deadline by Contract/Addendum',
 'Service',
 'Terms Requirements',
 'Contract Available',
 'Monitoring',
 'Delivery Status',
 'BAST Status',
 'BAST Sent',
 'BAST Received',
 'Baseline delivery Date',
 'Plan delivery Date',
 'Delayed Days',
 'Latest Update',
 'Term Status',
 'Invoice Sent',
 'Payment Estimate',
 'Payment Received',
 'Total Target Invoice',
 'Overdue',
 'Apr-26',
 'May-26',
 'Jun-26',
 'Jul-26',
 'Unscheduled',
 'On Hold',
 'ADM Status',
 'Cutoff KPI',
 'Co-PM',
 'Co-PD',
 'GTG-D',
 'THD-D',
 'TSR-D',
 'SSM-D',
 'DGS-D',
 'MKR-D',
 'RRR-D',
 'NAR-D',
 'CYH-D',
 'DDM-D',
 'AAM-D',
 'JRM-D',
 'NAW-D',
 'HZP-D',
 'ABH-D',
 'DTP-D',
 'IFI-D',
 'IMC-D',
 'HZR-D',
 'Unmapped',
 'GTG-D.1',
 'THD-D.1',
 'TSR-D.1',
 'SSM-D.1',
 'DGS-D.1',
 'MKR-D.1',
 'RRR-D.1',
 'NAR-D.1',
 'CYH-D.1',
 'DDM-D.1',
 'AAM-D.1',
 'JRM-D.1',
 'NAW-D.1',
 'HZP-D.1',
 'ABH-D.1',
 '

In [46]:
# replace missing values
df = df.replace([
    '-',
    '#N/A',
    '',
    ' '
], pd.NA)

In [47]:
# convert date columns

date_cols = [
    'SPK Date',
    'Deadline by Contract/Addendum',
    'BAST Sent',
    'BAST Received',
    'Invoice Sent',
    'Payment Estimate',
    'Payment Received'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [48]:
#inspect term column
sorted(df['Term'].dropna().unique())

['DP',
 'OPE',
 'T01',
 'T02',
 'T03',
 'T04',
 'T05',
 'T06',
 'T07',
 'T08',
 'T09',
 'T1',
 'T10',
 'T11',
 'T12',
 'T13',
 'T14',
 'T15',
 'T16',
 'T17',
 'T18',
 'T19',
 'T20',
 'T21',
 'T22',
 'T23',
 'T24',
 'T25',
 'T26',
 'T27',
 'T28',
 'T29',
 'T30',
 'T301',
 'T302',
 'T303',
 'T304',
 'T31',
 'T4',
 'T401',
 'T403',
 'T404',
 'T501',
 'T502',
 'T503',
 'T504',
 'gift',
 'printing',
 't01',
 't02',
 't03',
 't04',
 't05',
 't06',
 't07',
 't08',
 't09',
 't1',
 't10',
 't101',
 't102',
 't103',
 't11',
 't12',
 't13',
 't14',
 't15',
 't16',
 't17',
 't18',
 't19',
 't2',
 't20',
 't201',
 't202',
 't21',
 't22',
 't23',
 't24',
 't3',
 't301',
 't302',
 't4',
 't401',
 't402',
 't4a',
 't4b',
 't5',
 't501',
 't502',
 't6',
 't601',
 't602',
 't7',
 't8',
 't9']

In [49]:
# convert Term from t1, t2, ... to 1, 2, ...

def clean_term(term):
    if pd.isna(term):
        return pd.NA

    term = str(term).strip().upper()

    # Find the first sequence of digits anywhere in the string
    match = re.search(r'\d+', term)

    if not match:
        return pd.NA

    digits = match.group()

    if len(digits) <= 2:
        return int(digits)

    return int(digits[0])

df['Term'] = df['Term'].apply(clean_term).astype('Int64')

df.head()

,Job Code,Client Code,Year,PD,PM,Work Status,Client,Subject,Term,Fee,...,NAW-D.1,HZP-D.1,ABH-D.1,DTP-D.1,IFI-D.1,IMC-D.1,HZR-D.1,Service Group,Count Check,Jobcode V2
0,BCDR007,KPK01,2013,AOW,NVN,2,Komisi Pemberantasan Korupsi,BCP Development (training dan menyusun BIA),1,"123,690,000",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
1,BCDR008,IDS01,2013,AOW,MSM,2,Indosat Ooredoo,Consultant services for BCM Enhancement,1,"199,200,000",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
2,BCDR009,BCI01,2014,TSR,SHS,2,Bumiputera Sekuritas,BCP Review,1,"8,000,000",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
3,BCDR009,BCI01,2014,TSR,SHS,2,Bumiputera Sekuritas,BCP Review,2,"12,000,000",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
4,BCDR010,BNI01,2014,TSR,NVN,2,BNI Securities,BCP Review & VA,1,"19,000,000",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN


In [50]:
# Convert Fee and Total Target Invoice to numeric

money_cols = ['Fee', 'Total Target Invoice']

for col in money_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
    )

    df[col] = pd.to_numeric(df[col], errors='coerce')

df[['Fee', 'Total Target Invoice']].head(10)

,Fee,Total Target Invoice
0,123690000.0,NaN
1,199200000.0,NaN
2,8000000.0,NaN
3,12000000.0,NaN
4,19000000.0,NaN
5,19000000.0,NaN
6,18000000.0,NaN
7,18000000.0,NaN
8,29295000.0,NaN
9,15000000.0,NaN


In [51]:
# Convert status

status_map = {
    0: 'Starting',
    1: 'In Progress',
    2: 'Completed',
    -1: 'Cancelled'
}

status_cols = [
    'Work Status',
    'Term Status',
    'ADM Status',
    'BAST Status',

]

for col in status_cols:
    df[col] = df[col].map(status_map)


In [52]:
# Convert boolean 
df['Contract Available'] = df['Contract Available'].astype(int).astype(bool)

In [53]:
df.head()

,Job Code,Client Code,Year,PD,PM,Work Status,Client,Subject,Term,Fee,...,NAW-D.1,HZP-D.1,ABH-D.1,DTP-D.1,IFI-D.1,IMC-D.1,HZR-D.1,Service Group,Count Check,Jobcode V2
0,BCDR007,KPK01,2013,AOW,NVN,Completed,Komisi Pemberantasan Korupsi,BCP Development (training dan menyusun BIA),1,123690000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
1,BCDR008,IDS01,2013,AOW,MSM,Completed,Indosat Ooredoo,Consultant services for BCM Enhancement,1,199200000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
2,BCDR009,BCI01,2014,TSR,SHS,Completed,Bumiputera Sekuritas,BCP Review,1,8000000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
3,BCDR009,BCI01,2014,TSR,SHS,Completed,Bumiputera Sekuritas,BCP Review,2,12000000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN
4,BCDR010,BNI01,2014,TSR,NVN,Completed,BNI Securities,BCP Review & VA,1,19000000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN


In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5865 entries, 0 to 5864
Data columns (total 82 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Job Code                       5865 non-null   object        
 1   Client Code                    5865 non-null   object        
 2   Year                           5865 non-null   int64         
 3   PD                             5865 non-null   object        
 4   PM                             5834 non-null   object        
 5   Work Status                    5865 non-null   object        
 6   Client                         5865 non-null   object        
 7   Subject                        5863 non-null   object        
 8   Term                           5767 non-null   Int64         
 9   Fee                            5865 non-null   float64       
 10  SPK Date                       5341 non-null   datetime64[ns]
 11  Deadline by Contr

# Table 1. Contribution Partner

### Objective: Create a table that records each employee's role and percentage contribution for every project.

In [55]:
# create a list of all Job Codes
job_codes = (
    df[['Job Code']]
    .drop_duplicates()
    .reset_index(drop=True)
)

job_codes.head()

,Job Code
0,BCDR007
1,BCDR008
2,BCDR009
3,BCDR010
4,BCDR011


In [56]:
# Create the Employee Pool

employees = [
    'DTP','DLP','IFI','NFI','FYA',
    'TSR','OMW','TLW','WNA','IFS',
    'AMF','JQV','RLM','DGS','HZR'
]

In [57]:
# Generate Contribution Partner Table

contribution_partner = []

for job in job_codes['Job Code']:
    # PD
    pd_emp = random.choice(employees)

    has_copd = random.random() < 0.5

    if has_copd:
        copd_emp = random.choice([e for e in employees if e != pd_emp])

        split = random.choice([(70,30), (60,40)])

        contribution_partner.append({
            'Job Code': job,
            'Position': 'PD',
            'Employee': pd_emp,
            'Score Contribution (%)': split[0]
        })

        contribution_partner.append({
            'Job Code': job,
            'Position': 'Co-PD',
            'Employee': copd_emp,
            'Score Contribution (%)': split[1]
        })

    else:

        contribution_partner.append({
            'Job Code': job,
            'Position': 'PD',
            'Employee': pd_emp,
            'Score Contribution (%)': 100
        })


    # PM
    pm_emp = random.choice([e for e in employees if e != pd_emp])

    has_copm = random.random() < 0.5

    if has_copm:

        copm_emp = random.choice(
            [e for e in employees if e not in [pd_emp, pm_emp]]
        )

        split = random.choice([(70,30), (60,40)])

        contribution_partner.append({
            'Job Code': job,
            'Position': 'PM',
            'Employee': pm_emp,
            'Score Contribution (%)': split[0]
        })

        contribution_partner.append({
            'Job Code': job,
            'Position': 'Co-PM',
            'Employee': copm_emp,
            'Score Contribution (%)': split[1]
        })

    else:

        contribution_partner.append({
            'Job Code': job,
            'Position': 'PM',
            'Employee': pm_emp,
            'Score Contribution (%)': 100
        })

In [58]:
# Convert to DataFrame
contribution_partner = pd.DataFrame(contribution_partner)

In [59]:
# View table
contribution_partner.head(20)

,Job Code,Position,Employee,Score Contribution (%)
0,BCDR007,PD,AMF,60
1,BCDR007,Co-PD,RLM,40
2,BCDR007,PM,NFI,70
3,BCDR007,Co-PM,DGS,30
4,BCDR008,PD,AMF,100
5,BCDR008,PM,WNA,70
6,BCDR008,Co-PM,OMW,30
7,BCDR009,PD,DTP,70
8,BCDR009,Co-PD,FYA,30
9,BCDR009,PM,IFS,60


In [60]:
# Check primary key
contribution_partner.duplicated(
    subset=['Job Code', 'Position', 'Employee']
).sum()

0

In [61]:
# Check the contribution values
contribution_partner['Score Contribution (%)'].value_counts().sort_index()

Score Contribution (%)
30      970
40     1036
60     1036
70      970
100    2036
Name: count, dtype: int64

In [62]:
# Check that each Job sums to 100% for PD and PM
contribution_partner.groupby(
    ['Job Code', 'Position']
)['Score Contribution (%)'].sum()

Job Code        Position
BBN01-25-11-01  Co-PD        30
                Co-PM        40
                PD           70
                PM           60
BCA01-25-09-01  PD          100
                           ... 
VNI01-25-10-01  Co-PD        40
                PD           60
                PM          100
VNI01-25-10-02  PD          100
                PM          100
Name: Score Contribution (%), Length: 6048, dtype: int64

## Table 2. Time Contribution

### Objective: Record how much time each employee spent on a project.

In [75]:
# Copy Table 1. 
time_contribution = contribution_partner.copy()

In [76]:
# Rename the score column

time_contribution = time_contribution.rename(
    columns={
        'Score Contribution (%)':'Time Contribution (hours)'
    }
)

time_contribution.head()

,Job Code,Position,Employee,Time Contribution (hours)
0,BCDR007,PD,AMF,60
1,BCDR007,Co-PD,RLM,40
2,BCDR007,PM,NFI,70
3,BCDR007,Co-PM,DGS,30
4,BCDR008,PD,AMF,100


In [77]:
# Decide a constant of the total hours
TOTAL_HOURS = 40

# Replace the percentages with hours
time_contribution['Time Contribution (hours)'] = (
    time_contribution['Time Contribution (hours)']
    /100
    *TOTAL_HOURS
)

In [78]:
# Convert to integer
time_contribution['Time Contribution (hours)'] = (
    time_contribution['Time Contribution (hours)']
    .astype(int)
)

In [80]:
time_contribution.head(20)

,Job Code,Position,Employee,Time Contribution (hours)
0,BCDR007,PD,AMF,24
1,BCDR007,Co-PD,RLM,16
2,BCDR007,PM,NFI,28
3,BCDR007,Co-PM,DGS,12
4,BCDR008,PD,AMF,40
5,BCDR008,PM,WNA,28
6,BCDR008,Co-PM,OMW,12
7,BCDR009,PD,DTP,28
8,BCDR009,Co-PD,FYA,12
9,BCDR009,PM,IFS,24


In [81]:
# Check Primary key

time_contribution.duplicated(
    subset=['Job Code','Position','Employee']
).sum()

0

In [85]:
# Validate the logic

check = time_contribution.copy()

check['Role Group'] = check['Position'].replace({
    'Co-PD': 'PD',
    'Co-PM': 'PM'
})

check.groupby(
    ['Job Code', 'Role Group']
)['Time Contribution (hours)'].sum()

Job Code        Role Group
BBN01-25-11-01  PD            40
                PM            40
BCA01-25-09-01  PD            40
                PM            40
BCA01-25-11-01  PD            40
                              ..
TSOP040         PM            40
VNI01-25-10-01  PD            40
                PM            40
VNI01-25-10-02  PD            40
                PM            40
Name: Time Contribution (hours), Length: 4042, dtype: int64

## Table 3 — Legal Activity
### Objective: Records the legal document activities and their corresponding dates for each project.